In [1]:
%pip install deep-translator

Note: you may need to restart the kernel to use updated packages.


In [2]:

import re
import threading
import time
import tkinter as tk
from tkinter import messagebox, ttk

try:
    from deep_translator import GoogleTranslator
except ImportError:
    GoogleTranslator = None

try:
    import requests  # comes together with deep-translator
except ImportError:
    requests = None

# ---------- Languages for "Translate" mode ----------
LANGUAGES = {
    "Auto Detect": "auto",
    "English": "en",
    "Hindi": "hi",
    "Marathi": "mr",
    "Gujarati": "gu",
    "Bengali": "bn",
    "Tamil": "ta",
    "Telugu": "te",
    "Kannada": "kn",
    "Malayalam": "ml",
    "Punjabi": "pa",
    "Urdu": "ur",
    "Nepali": "ne",
    "French": "fr",
    "Spanish": "es",
    "German": "de",
    "Italian": "it",
    "Portuguese": "pt",
    "Russian": "ru",
    "Japanese": "ja",
    "Korean": "ko",
    "Chinese (Simplified)": "zh-CN",
    "Arabic": "ar",
    "Turkish": "tr",
}
TARGET_LANGUAGES = [name for name in LANGUAGES if name != "Auto Detect"]

# Source choices that mean "typed in English letters": name -> real language
HINGLISH = "Hinglish (Hindi in English letters)"
MARATHI_ROMAN = "Marathi in English letters"
ROMAN_SOURCES = {HINGLISH: "Hindi", MARATHI_ROMAN: "Marathi"}

SOURCE_LANGUAGES = (
    ["Auto Detect", HINGLISH, MARATHI_ROMAN]
    + [name for name in LANGUAGES if name != "Auto Detect"]
)

# ---------- Languages for "Names & Hinglish" mode ----------
CONVERT_CODES = {
    "Hindi": "hi",
    "Marathi": "mr",
    "Gujarati": "gu",
    "Bengali": "bn",
    "Tamil": "ta",
    "Telugu": "te",
    "Kannada": "kn",
    "Malayalam": "ml",
    "Punjabi": "pa",
    "Urdu": "ur",
    "Nepali": "ne",
}
CONVERT_LANGUAGES = list(CONVERT_CODES.keys())
CONVERT_URL = "https://inputtools.google.com/request"
GTX_URL = "https://translate.googleapis.com/translate_a/single"
WORDS_PER_REQUEST = 10
ROMAN_LABEL = "English letters (a, b, c)"

# Example words written as unicode so the file works on any computer:
#   HELLO_HI = Hindi word for "hello", RIYA_HI = Riya, KAISE_HO_HI = kaise ho
HELLO_HI = "नमस्ते"
RIYA_HI = "रिया"
KAISE_HO_HI = "कैसे हो"

HINT_TRANSLATE = (
    f"Changes the meaning into another language.   Example: Hello → {HELLO_HI}\n"
    "Tip: pick the source language yourself (Auto Detect often guesses wrong for names). "
    "Typing Hindi in English letters? Choose Hinglish."
)
HINT_CONVERT = (
    "Changes only the letters, not the meaning (use this for names).\n"
    f"Example: Riya → {RIYA_HI},   kaise ho → {KAISE_HO_HI}"
)

# ---------- Colors & fonts ----------
BG = "#f4f6fb"
CARD = "#ffffff"
ACCENT = "#4f46e5"
ACCENT_HOVER = "#4338ca"
TEXT = "#1f2937"
MUTED = "#6b7280"
BORDER = "#d9deea"
RED = "#dc2626"
GREEN = "#16a34a"
FONT = ("Segoe UI", 11)
FONT_BOLD = ("Segoe UI", 11, "bold")
FONT_TITLE = ("Segoe UI", 20, "bold")

# ---------- Settings ----------
MAX_CHARS = 5000          # longest text allowed
CHUNK_CHARS = 1000        # long text is sent in pieces of this size
RETRY_WAITS = (2,)        # one short wait, then one more try

# ---------- Friendly error messages: (short status, full explanation) ----------
MESSAGES = {
    "rate": (
        "Google is busy. Try again in a minute.",
        "Google is limiting requests from your internet connection right now."
        "\n\nWhat to do:\n1. Wait about one minute, then press the button again."
        "\n2. Still not working? Switch your internet (for example, turn on your "
        "phone's hotspot) and try again.\n\nThis happens when many people share "
        "one internet connection.",
    ),
    "network": (
        "No internet connection.",
        "Could not reach the server.\n\nCheck that your Wi-Fi or mobile data is "
        "working, then press the button again.",
    ),
    "language": (
        "Language not supported.",
        "This language pair is not supported.\n\nTry choosing a different "
        "language.",
    ),
    "length": (
        "Text is too long.",
        "The text is too long for one request.\n\nShorten it and try again.",
    ),
    "other": (
        "Something went wrong.",
        "Something went wrong.\n\nPress the button again. If it keeps failing, "
        "wait a minute and retry.",
    ),
}

NETWORK_WORDS = (
    "connection",
    "timed out",
    "timeout",
    "max retries",
    "name resolution",
    "getaddrinfo",
    "unreachable",
    "network",
    "ssl",
)


# =====================================================================
#  Logic (no window code here)
# =====================================================================
def classify_error(error):
    """Turn any error into one of: rate, network, language, length, other."""
    name = type(error).__name__.lower()
    text = str(error).lower()

    if "toomanyrequests" in name or "too many requests" in text or "429" in text:
        return "rate"
    if "notvalidlength" in name:
        return "length"
    if "languagenotsupported" in name or "not supported" in text:
        return "language"
    if any(word in text for word in NETWORK_WORDS) or any(
        word in name for word in ("connection", "timeout")
    ):
        return "network"
    return "other"


# ---------- Mode 2 first (also used by Hinglish): letters -> other letters ----------
def convert_chunk_once(chunk, code):
    """Ask Google Input Tools to write `chunk` (English letters) in another script."""
    if requests is None:
        raise RuntimeError("the 'requests' package is missing")

    response = requests.get(
        CONVERT_URL,
        params={
            "text": chunk,
            "itc": f"{code}-t-i0-und",
            "num": 1,
            "cp": 0,
            "cs": 1,
            "ie": "utf-8",
            "oe": "utf-8",
            "app": "demopage",
        },
        timeout=10,
    )
    response.raise_for_status()
    data = response.json()

    try:
        if data[0] != "SUCCESS":
            raise ValueError("server did not return SUCCESS")
        converted = data[1][0][1][0]
    except (IndexError, KeyError, TypeError):
        raise ValueError("unexpected answer from the server")

    if not str(converted).strip():
        raise ValueError("empty answer from the server")
    return str(converted)


def convert_chunk(chunk, code, on_progress, sleep):
    """Same as above, but waits and retries if Google says 'too many requests'."""
    attempts = len(RETRY_WAITS) + 1
    for attempt in range(1, attempts + 1):
        try:
            return convert_chunk_once(chunk, code)
        except Exception as error:
            if classify_error(error) == "rate" and attempt < attempts:
                if on_progress:
                    on_progress("Google is busy. Trying once more...")
                sleep(RETRY_WAITS[attempt - 1])
                continue
            raise


def run_conversion(text, target_name, on_progress=None, sleep=time.sleep):
    """
    Writes English-letter text (names, Hinglish) in the letters of the target
    language. Keeps line breaks.
    Returns (converted_text, None) or (None, error_kind).
    """
    code = CONVERT_CODES[target_name]
    out_lines = []
    first_request = True

    try:
        for line in text.split("\n"):
            words = line.split()
            if not words:
                out_lines.append("")
                continue

            pieces = []
            for start in range(0, len(words), WORDS_PER_REQUEST):
                if not first_request:
                    sleep(0.3)  # be gentle with the server
                first_request = False
                chunk = " ".join(words[start:start + WORDS_PER_REQUEST])
                pieces.append(convert_chunk(chunk, code, on_progress, sleep))
            out_lines.append(" ".join(pieces))
    except Exception as error:
        return None, classify_error(error)

    return "\n".join(out_lines), None


# ---------- Mode 1: Translate (meaning) ----------
def translate_gtx(text, source, target):
    """Route 1: Google's direct translate address (usually the friendliest)."""
    if requests is None:
        raise RuntimeError("the 'requests' package is missing")

    response = requests.get(
        GTX_URL,
        params={"client": "gtx", "sl": source, "tl": target, "dt": "t", "q": text},
        timeout=10,
    )
    response.raise_for_status()
    data = response.json()

    try:
        pieces = [
            part[0] for part in data[0] if part and isinstance(part[0], str)
        ]
    except (IndexError, KeyError, TypeError):
        raise ValueError("unexpected answer from the server")

    result = "".join(pieces)
    if not result.strip():
        raise ValueError("empty translation")
    return result


def translate_google_page(text, source, target):
    """Route 2: the translate page used by the deep-translator package."""
    if GoogleTranslator is None:
        raise RuntimeError("the 'deep-translator' package is missing")

    result = GoogleTranslator(source=source, target=target).translate(text)
    if result is None or not str(result).strip():
        raise ValueError("empty translation")
    return str(result)


def translate_chunk(chunk, source, target, on_progress, sleep):
    """
    Tries route 1, then route 2 right away (no waiting between them).
    Only if Google says "too many requests" does it wait a moment and go once more.
    Returns (text, None) or (None, error_kind).
    """
    routes = [translate_gtx, translate_google_page]
    kinds = []

    for round_number in range(2):
        kinds = []
        for route in routes:
            try:
                return route(chunk, source, target), None
            except Exception as error:
                kinds.append(classify_error(error))

        if "rate" not in kinds or round_number == 1:
            break
        if on_progress:
            on_progress("Google is busy. Trying once more...")
        sleep(RETRY_WAITS[0])

    for kind in ("rate", "language", "length", "network"):
        if kind in kinds:
            return None, kind
    return None, "other"


def split_text(text, limit=CHUNK_CHARS):
    """
    Cuts long text into pieces of at most `limit` characters, only at spaces or
    line breaks. Joining the pieces gives back exactly the original text.
    """
    chunks = []
    current = ""
    for token in re.split(r"(\s+)", text):
        while len(token) > limit:  # one gigantic word: cut it
            if current:
                chunks.append(current)
                current = ""
            chunks.append(token[:limit])
            token = token[limit:]
        if current and len(current) + len(token) > limit:
            chunks.append(current)
            current = ""
        current += token
    if current:
        chunks.append(current)
    return chunks


def run_translation(text, source_name, target_name, on_progress=None, sleep=time.sleep):
    """
    Returns (translated_text, None) on success,
    or (None, error_kind) when everything failed.
    """
    # Hinglish: first write it in real Hindi letters, then translate from Hindi
    if source_name in ROMAN_SOURCES:
        script_language = ROMAN_SOURCES[source_name]
        text, error_kind = run_conversion(text, script_language, on_progress, sleep)
        if text is None:
            return None, error_kind
        if target_name == script_language:
            return text, None
        source = LANGUAGES[script_language]
    else:
        source = LANGUAGES[source_name]

    target = LANGUAGES[target_name]

    parts = []
    for chunk in split_text(text):
        core = chunk.strip()
        if not core:
            parts.append(chunk)
            continue

        lead = chunk[: len(chunk) - len(chunk.lstrip())]
        trail = chunk[len(chunk.rstrip()):]

        result, error_kind = translate_chunk(core, source, target, on_progress, sleep)
        if result is None:
            return None, error_kind
        parts.append(lead + result.strip() + trail)

    return "".join(parts), None


# =====================================================================
#  The window
# =====================================================================
class TranslatorApp:
    def __init__(self, root):
        self.root = root
        root.title("Language Translator")
        root.geometry("900x640")
        root.minsize(760, 560)
        root.configure(bg=BG)

        self.busy = False
        self.last_result = ""
        self.cache = {}
        self.generation = 0          # goes up every time Clear is pressed
        self.saved_source = "English"

        self.setup_styles()
        self.build_ui()
        self.bind_shortcuts()

    # ---------- Styling ----------
    def setup_styles(self):
        style = ttk.Style()
        style.theme_use("clam")

        style.configure("TFrame", background=BG)

        style.configure(
            "TCombobox",
            fieldbackground=CARD,
            background=CARD,
            foreground=TEXT,
            bordercolor=BORDER,
            arrowcolor=ACCENT,
            padding=6,
        )
        style.map(
            "TCombobox",
            fieldbackground=[("readonly", CARD), ("disabled", "#eceef6")],
            foreground=[("disabled", MUTED)],
        )

        style.configure(
            "Mode.TRadiobutton", background=BG, foreground=TEXT, font=FONT_BOLD
        )
        style.map("Mode.TRadiobutton", background=[("active", BG)])

        style.configure(
            "Primary.TButton",
            background=ACCENT,
            foreground="white",
            font=FONT_BOLD,
            padding=(26, 10),
            borderwidth=0,
        )
        style.map(
            "Primary.TButton",
            background=[("active", ACCENT_HOVER), ("disabled", "#a5a3e8")],
        )

        style.configure(
            "Danger.TButton",
            background="#fde8e8",
            foreground=RED,
            font=FONT_BOLD,
            padding=(26, 10),
            borderwidth=0,
        )
        style.map("Danger.TButton", background=[("active", "#fbd0d0")])

        style.configure(
            "Soft.TButton",
            background="#e8eaf6",
            foreground=ACCENT,
            font=FONT_BOLD,
            padding=(16, 10),
            borderwidth=0,
        )
        style.map("Soft.TButton", background=[("active", "#d9dcf2")])

        style.configure(
            "Swap.TButton",
            background=BG,
            foreground=ACCENT,
            font=("Segoe UI", 16, "bold"),
            padding=(10, 2),
            borderwidth=0,
        )
        style.map("Swap.TButton", background=[("active", "#e8eaf6")])

    # ---------- Layout ----------
    def build_ui(self):
        outer = ttk.Frame(self.root, padding=20)
        outer.pack(fill="both", expand=True)

        tk.Label(
            outer, text="\U0001F310 Language Translator", font=FONT_TITLE, bg=BG, fg=TEXT
        ).pack(anchor="w")
        tk.Label(
            outer,
            text="Type on the left, press the button (or Ctrl + Enter).",
            font=FONT,
            bg=BG,
            fg=MUTED,
        ).pack(anchor="w", pady=(0, 10))

        # Mode switch
        mode_row = ttk.Frame(outer)
        mode_row.pack(fill="x")
        tk.Label(mode_row, text="Mode:", font=FONT_BOLD, bg=BG, fg=TEXT).pack(
            side="left", padx=(0, 10)
        )

        self.mode_var = tk.StringVar(value="translate")
        self.mode_buttons = [
            ttk.Radiobutton(
                mode_row,
                text="Translate",
                value="translate",
                variable=self.mode_var,
                command=self.on_mode_change,
                style="Mode.TRadiobutton",
            ),
            ttk.Radiobutton(
                mode_row,
                text="Names & Hinglish",
                value="convert",
                variable=self.mode_var,
                command=self.on_mode_change,
                style="Mode.TRadiobutton",
            ),
        ]
        self.mode_buttons[0].pack(side="left", padx=(0, 18))
        self.mode_buttons[1].pack(side="left")

        self.hint = tk.Label(
            outer,
            text=HINT_TRANSLATE,
            font=("Segoe UI", 10),
            bg=BG,
            fg=MUTED,
            anchor="w",
            justify="left",
        )
        self.hint.pack(fill="x", pady=(2, 12))

        # Language selector row
        lang_row = ttk.Frame(outer)
        lang_row.pack(fill="x", pady=(0, 10))
        lang_row.columnconfigure(0, weight=1)
        lang_row.columnconfigure(2, weight=1)

        self.src_var = tk.StringVar(value="English")
        self.tgt_var = tk.StringVar(value="Hindi")

        self.src_box = ttk.Combobox(
            lang_row,
            textvariable=self.src_var,
            values=SOURCE_LANGUAGES,
            state="readonly",
            font=FONT,
        )
        self.src_box.grid(row=0, column=0, sticky="ew")

        ttk.Button(
            lang_row, text="⇄", style="Swap.TButton", command=self.swap_languages
        ).grid(row=0, column=1, padx=10)

        self.tgt_box = ttk.Combobox(
            lang_row,
            textvariable=self.tgt_var,
            values=TARGET_LANGUAGES,
            state="readonly",
            font=FONT,
        )
        self.tgt_box.grid(row=0, column=2, sticky="ew")

        # Remove the blue highlight after choosing a language
        for box in (self.src_box, self.tgt_box):
            box.bind("<<ComboboxSelected>>", lambda e: e.widget.selection_clear())

        # ---- Buttons row ----
        # Packed BEFORE the text panels and pinned to the bottom, so it is
        # always visible, even when the window is small.
        btn_row = ttk.Frame(outer)
        btn_row.pack(side="bottom", fill="x", pady=(14, 0))

        self.translate_btn = ttk.Button(
            btn_row, text="Translate", style="Primary.TButton", command=self.translate
        )
        self.translate_btn.pack(side="left")

        ttk.Button(
            btn_row, text="Clear", style="Danger.TButton", command=self.clear_all
        ).pack(side="left", padx=10)

        ttk.Button(
            btn_row, text="Copy result", style="Soft.TButton", command=self.copy_result
        ).pack(side="left")

        self.status = tk.Label(
            btn_row, text="Ready", font=FONT, bg=BG, fg=MUTED, anchor="e"
        )
        self.status.pack(side="right")

        # ---- Text panels (fill whatever space is left) ----
        panels = ttk.Frame(outer)
        panels.pack(fill="both", expand=True)
        panels.columnconfigure(0, weight=1, uniform="p")
        panels.columnconfigure(1, weight=1, uniform="p")
        panels.rowconfigure(0, weight=1)

        self.input_box, self.counter = self.make_panel(panels, 0, editable=True)
        self.output_box, _ = self.make_panel(panels, 1, editable=False)
        self.output_box.tag_configure("error", foreground=RED)

    def make_panel(self, parent, column, editable):
        card = tk.Frame(
            parent, bg=CARD, highlightbackground=BORDER, highlightthickness=1
        )
        card.grid(
            row=0,
            column=column,
            sticky="nsew",
            padx=(0, 8) if column == 0 else (8, 0),
        )

        text = tk.Text(
            card,
            wrap="word",
            height=6,
            font=("Segoe UI", 12),
            bg=CARD,
            fg=TEXT,
            relief="flat",
            padx=14,
            pady=12,
            insertbackground=ACCENT,
            undo=True,
        )

        counter = None
        if editable:
            # Counter first (pinned to the bottom), then the text area above it
            counter = tk.Label(
                card,
                text=f"0 / {MAX_CHARS}",
                font=("Segoe UI", 9),
                bg=CARD,
                fg=MUTED,
                anchor="e",
            )
            counter.pack(side="bottom", fill="x", padx=12, pady=(0, 6))
            text.pack(fill="both", expand=True)
            text.bind("<KeyRelease>", self.update_counter)
            text.bind("<<Paste>>", lambda e: self.root.after(50, self.update_counter))
            text.focus_set()
        else:
            text.pack(fill="both", expand=True)
            text.configure(state="disabled", bg="#fafbff")

        return text, counter

    # ---------- Shortcuts ----------
    def bind_shortcuts(self):
        self.root.bind("<Control-Return>", self.on_ctrl_enter)
        self.input_box.bind("<Control-Return>", self.on_ctrl_enter)

    def on_ctrl_enter(self, event=None):
        self.translate()
        return "break"

    # ---------- Small helpers ----------
    def call_later(self, func, *args):
        """Run something on the main window thread (safe even if window closed)."""
        try:
            self.root.after(0, func, *args)
        except Exception:
            pass

    def update_counter(self, event=None):
        length = len(self.input_box.get("1.0", "end").strip())
        self.counter.config(
            text=f"{length} / {MAX_CHARS}", fg=RED if length > MAX_CHARS else MUTED
        )

    def set_status(self, message, color=MUTED):
        self.status.config(text=message, fg=color)

    def set_output(self, text, is_error=False):
        self.output_box.configure(state="normal")
        self.output_box.delete("1.0", "end")
        if is_error:
            self.output_box.insert("1.0", text, "error")
        else:
            self.output_box.insert("1.0", text)
        self.output_box.configure(state="disabled")

    def set_mode_buttons(self, enabled):
        for button in self.mode_buttons:
            button.state(["!disabled"] if enabled else ["disabled"])

    # ---------- Mode switch ----------
    def on_mode_change(self):
        if self.busy:
            return

        if self.mode_var.get() == "convert":
            self.saved_source = self.src_var.get()
            self.src_var.set(ROMAN_LABEL)
            self.src_box.configure(state="disabled")
            self.tgt_box.configure(values=CONVERT_LANGUAGES)
            if self.tgt_var.get() not in CONVERT_LANGUAGES:
                self.tgt_var.set("Hindi")
            self.translate_btn.config(text="Convert")
            self.hint.config(text=HINT_CONVERT)
        else:
            self.src_box.configure(state="readonly")
            self.src_var.set(self.saved_source)
            self.tgt_box.configure(values=TARGET_LANGUAGES)
            self.translate_btn.config(text="Translate")
            self.hint.config(text=HINT_TRANSLATE)

        self.last_result = ""
        self.set_output("")
        self.set_status("Ready")

    # ---------- Main action ----------
    def translate(self):
        if self.busy:
            return

        text = self.input_box.get("1.0", "end").strip()
        if not text:
            self.set_status("Please type something first.", RED)
            return
        if len(text) > MAX_CHARS:
            self.set_status(f"Too long (max {MAX_CHARS} characters).", RED)
            return

        mode = self.mode_var.get()
        source_name = self.src_var.get() if mode == "translate" else ""
        target_name = self.tgt_var.get()

        # Same text asked before? Show it instantly, no request needed.
        key = (mode, text, source_name, target_name)
        if key in self.cache:
            self.show_result(self.cache[key], "Done ✓")
            return

        self.busy = True
        self.last_result = ""
        self.translate_btn.state(["disabled"])
        self.set_mode_buttons(False)
        self.set_status("Working...", ACCENT)

        # Background thread so the window never freezes
        threading.Thread(
            target=self.worker, args=(key, self.generation), daemon=True
        ).start()

    def worker(self, key, generation):
        mode, text, source_name, target_name = key

        def progress(message):
            self.call_later(self.set_status, message, ACCENT)

        try:
            if mode == "convert":
                result, error_kind = run_conversion(text, target_name, progress)
            else:
                result, error_kind = run_translation(
                    text, source_name, target_name, progress
                )
        except Exception:
            result, error_kind = None, "other"

        self.call_later(self.finish, key, generation, result, error_kind)

    def finish(self, key, generation, result, error_kind):
        self.busy = False
        self.translate_btn.state(["!disabled"])
        self.set_mode_buttons(True)

        # Clear was pressed while this was running -> throw the answer away
        if generation != self.generation:
            return

        if result is not None:
            self.cache[key] = result
            self.show_result(result, "Done ✓")
        else:
            short, full = MESSAGES.get(error_kind, MESSAGES["other"])
            self.last_result = ""
            self.set_output(full, is_error=True)
            self.set_status(short, RED)

    def show_result(self, result, status):
        self.last_result = result
        self.set_output(result)
        self.set_status(status, GREEN)

    # ---------- Other buttons ----------
    def swap_languages(self):
        if self.mode_var.get() == "convert":
            self.set_status("Swap works only in Translate mode.", RED)
            return

        src = self.src_var.get()
        if src == "Auto Detect" or src in ROMAN_SOURCES:
            self.set_status("Swap needs a normal source language.", RED)
            return

        tgt = self.tgt_var.get()
        self.src_var.set(tgt)
        self.tgt_var.set(src)

        # Move the translated text back into the input box
        if self.last_result:
            self.input_box.delete("1.0", "end")
            self.input_box.insert("1.0", self.last_result)
            self.last_result = ""
            self.set_output("")
            self.update_counter()

    def copy_result(self):
        if not self.last_result:
            self.set_status("Nothing to copy yet.", RED)
            return
        self.root.clipboard_clear()
        self.root.clipboard_append(self.last_result)
        self.set_status("Copied to clipboard ✓", GREEN)

    def clear_all(self):
        """Wipe everything: your text, the result, and all remembered results."""
        self.generation += 1          # any answer still on its way is ignored
        self.cache.clear()
        self.last_result = ""
        self.input_box.delete("1.0", "end")
        self.set_output("")
        self.update_counter()
        self.set_status("Cleared ✓", GREEN)
        self.input_box.focus_set()


if __name__ == "__main__":
    root = tk.Tk()
    if GoogleTranslator is None:
        root.withdraw()
        messagebox.showerror(
            "Missing package",
            "The package 'deep-translator' is not installed.\n\n"
            "Open the terminal in VS Code and run:\n\n"
            "pip install deep-translator\n\n"
            "Then run this app again.",
        )
        root.destroy()
    else:
        TranslatorApp(root)
        root.mainloop()